In [1]:
from contextlib import nullcontext
import numpy as np
import torch

from App_utils.pre_processing import normalize_mri_to_uint8, resize_grayscale_stack_to_rgb


def run_medsam_inf_return_conf(
    vol: np.ndarray,
    predictor,
    image_size: int,
    prompts_by_slice: dict[int, tuple[np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None]],
    p_low: float = 1.0,
    p_high: float = 99.0,
    threshold: float = 0.0,
    propagation_style: str = "default",
):
    """
    Run MedSAM2 inference on a 3D MRI volume using prompts on selected slices.

    Returns
    -------
    probs_3d : np.ndarray
        Soft foreground probabilities with shape (D, H, W), dtype float32.
        Each voxel is in [0, 1] and represents the model's foreground score
        after sigmoid.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    torch.manual_seed(1604)
    np.random.seed(1604)

    if device.type == "cuda":
        torch.cuda.manual_seed(1604)

    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {vol.shape}")

    D, H, W = vol.shape
    print("Volume shape (D,H,W):", (D, H, W))

    vol_u8 = normalize_mri_to_uint8(vol, p_low=p_low, p_high=p_high)
    frames = resize_grayscale_stack_to_rgb(vol_u8, image_size)

    frames_t = torch.from_numpy(frames).to(device)
    img_mean = torch.tensor((0.485, 0.456, 0.406), dtype=torch.float32, device=device)[:, None, None]
    img_std = torch.tensor((0.229, 0.224, 0.225), dtype=torch.float32, device=device)[:, None, None]
    frames_t = (frames_t - img_mean) / img_std

    if hasattr(predictor, "to"):
        predictor = predictor.to(device)

    if hasattr(predictor, "model") and hasattr(predictor.model, "to"):
        predictor.model = predictor.model.to(device)

    autocast_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        if device.type == "cuda"
        else nullcontext()
    )

    probs_3d = np.zeros((D, H, W), dtype=np.float32)

    def has_valid_prompt(points, point_labels, bbox, mask_input):
        has_points = points is not None and point_labels is not None and len(points) > 0
        has_box = bbox is not None
        has_mask = mask_input is not None and mask_input.sum() > 0
        return has_points or has_box or has_mask

    def add_prompt_for_slice(inference_state, slice_idx, points, point_labels, bbox, mask_input):
        has_points = points is not None and point_labels is not None and len(points) > 0
        has_box = bbox is not None
        has_mask = mask_input is not None and mask_input.sum() > 0

        print(f"Adding prompt(s) on slice {slice_idx}")

        if has_mask:
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=slice_idx,
                obj_id=1,
                mask=mask_input,
            )

        if has_points and has_box:
            predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=slice_idx,
                obj_id=1,
                points=points,
                labels=point_labels,
                box=bbox,
            )
        elif has_points:
            predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=slice_idx,
                obj_id=1,
                points=points,
                labels=point_labels,
            )
        elif has_box:
            predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=slice_idx,
                obj_id=1,
                box=bbox,
            )
        elif not has_mask:
            raise ValueError(f"No valid prompts found for slice {slice_idx}")

    valid_slice_indices = [
        slice_idx
        for slice_idx in sorted(prompts_by_slice.keys())
        if has_valid_prompt(*prompts_by_slice[slice_idx])
    ]

    if len(valid_slice_indices) == 0:
        raise ValueError("No usable prompts found on any slice.")

    with torch.inference_mode(), autocast_ctx:
        inference_state = predictor.init_state(frames_t, H, W)

        print("Using prompts from slices:", valid_slice_indices)

        for slice_idx in valid_slice_indices:
            points, point_labels, bbox, mask_input = prompts_by_slice[slice_idx]
            add_prompt_for_slice(inference_state, slice_idx, points, point_labels, bbox, mask_input)

        if propagation_style == "default":
            print("Forward propagation (default)...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

            print("Backward propagation (default)...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state,
                reverse=True,
            ):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

        elif propagation_style == "full":
            print("Forward propagation (full, from slice 0)...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state,
                start_frame_idx=0,
            ):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

            print(f"Backward propagation (full, from slice {D - 1})...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state,
                start_frame_idx=D - 1,
                reverse=True,
            ):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

        elif propagation_style == "prompt_based":
            start_fwd = min(valid_slice_indices)
            start_bwd = max(valid_slice_indices)

            print(f"Forward propagation (prompt_based, from slice {start_fwd})...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state,
                start_frame_idx=start_fwd,
            ):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

            print(f"Backward propagation (prompt_based, from slice {start_bwd})...")
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state,
                start_frame_idx=start_bwd,
                reverse=True,
            ):
                prob2d = torch.sigmoid(out_mask_logits[0]).detach().cpu().numpy()[0].astype(np.float32)
                probs_3d[out_frame_idx] = np.maximum(probs_3d[out_frame_idx], prob2d)

        else:
            raise ValueError(
                f"Unknown propagation_style '{propagation_style}'. "
                "Choose from: 'default', 'full', 'prompt_based'"
            )

        predictor.reset_state(inference_state)

    return probs_3d

In [2]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk

from sam2.build_sam import build_sam2_video_predictor_npz
from App_utils.prompt_utils import load_mask_like_reference


def confidence_vis_pipeline(
    input_folder,
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    use_dense_mask=True,
    use_bbox=False,
    use_center_point=False,
    use_negative_background_points=False,   # NEW
    propagation_style="default",
    verbose=False,
    subject_nr=0,
):
    """
    Run MedSAM2 on one selected subject and return the confidence map.

    Prompt strategies
    -----------------
    dense_mask:
        Use the dense mask on each slice as mask prompt.

    bbox:
        Convert the dense mask on each slice to a bounding box prompt.

    center_point:
        Place one positive point at the center of the dense mask on each slice.

    negative_background_points:
        Add fixed negative points far from the dense mask, based on image-relative
        positions (for example around the 25%/75% locations).
    """

    root = Path(input_folder)
    if not root.is_dir():
        raise FileNotFoundError(f"Input folder does not exist: {root}")

    subjects = sorted([p.name for p in root.iterdir() if p.is_dir()])
    if len(subjects) == 0:
        raise ValueError(f"No subject folders found in: {root}")

    if not (0 <= subject_nr < len(subjects)):
        raise IndexError(f"subject_nr must be between 0 and {len(subjects)-1}, got {subject_nr}")

    if verbose:
        print(f"Building SAM predictor from checkpoint: {checkpoint}")

    predictor = build_sam2_video_predictor_npz(cfg, checkpoint)

    subject = subjects[subject_nr]
    subject_path = root / subject / "MR_StorT2"
    img_path = subject_path / "image.nii.gz"

    if not img_path.is_file():
        raise FileNotFoundError(f"Missing image: {img_path}")

    if volume_of_interest == "CTVT":
        dense_prompt_path = subject_path / "nnUNetOutput" / "mask_CTVT_427_nnUNet.nii.gz"
        gt_path = subject_path / "mask_CTVT_427.nii.gz"
    elif volume_of_interest == "rectum":
        dense_prompt_path = subject_path / "nnUNetOutput" / "mask_Rectum_nnUNet.nii.gz"
        gt_path = subject_path / "mask_Rectum.nii.gz"
    else:
        raise ValueError("volume_of_interest must be 'CTVT' or 'rectum'")

    if not dense_prompt_path.is_file():
        raise FileNotFoundError(f"Missing dense prompt: {dense_prompt_path}")

    if verbose:
        print(f"Processing {subject}")

    img_itk = sitk.ReadImage(str(img_path))
    img = sitk.GetArrayFromImage(img_itk).astype(np.float32)

    dense_mask_3d = load_mask_like_reference(str(dense_prompt_path), img.shape)
    gt_mask = load_mask_like_reference(str(gt_path), img.shape) if gt_path.is_file() else None

    prompts_by_slice = build_prompts_by_slice_from_dense_mask(
        dense_mask_3d=dense_mask_3d,
        use_dense_mask=use_dense_mask,
        use_bbox=use_bbox,
        use_center_point=use_center_point,
        use_negative_background_points=use_negative_background_points,   # NEW
    )

    probs_3d = run_medsam_inf_return_conf(
        vol=img,
        predictor=predictor,
        image_size=image_size,
        prompts_by_slice=prompts_by_slice,
        p_low=p_low,
        p_high=p_high,
        threshold=0.0,
        propagation_style=propagation_style,
    )

    confidence_3d = 2 * np.abs(probs_3d - 0.5)

    conf_itk = sitk.GetImageFromArray(confidence_3d.astype(np.float32))
    conf_itk.CopyInformation(img_itk)
    sitk.WriteImage(
        conf_itk,
        str(
            subject_path
            / f"confidence_DenseBboxPointNeg_"
              f"{use_dense_mask}{use_bbox}{use_center_point}{use_negative_background_points}.nii.gz"
        )
    )

    return img, confidence_3d


def build_prompts_by_slice_from_dense_mask(
    dense_mask_3d,
    use_dense_mask=True,
    use_bbox=False,
    use_center_point=False,
    use_negative_background_points=False,   # NEW
):
    prompts_by_slice = {}

    dense_slices = np.where(
        dense_mask_3d.reshape(dense_mask_3d.shape[0], -1).sum(axis=1) > 0
    )[0].astype(int)

    for slice_idx in dense_slices:
        mask2d = dense_mask_3d[slice_idx].astype(np.uint8)

        points_list = []
        labels_list = []

        bbox = None
        mask_input = mask2d if use_dense_mask else None

        if use_bbox:
            bbox = mask_to_bbox_xyxy(mask2d)

        if use_center_point:
            point = mask_to_center_point(mask2d)
            if point is not None:
                points_list.append(point.astype(np.float32))
                labels_list.append(1)

        if use_negative_background_points:
            neg_points = mask_to_negative_points_far_outside(mask2d)
            if neg_points is not None and len(neg_points) > 0:
                for p in neg_points:
                    points_list.append(p.astype(np.float32))
                    labels_list.append(0)

        points = None
        point_labels = None
        if len(points_list) > 0:
            points = np.stack(points_list, axis=0).astype(np.float32)
            point_labels = np.array(labels_list, dtype=np.int64)

        if mask_input is None and bbox is None and points is None:
            continue

        prompts_by_slice[int(slice_idx)] = (points, point_labels, bbox, mask_input)

    if len(prompts_by_slice) == 0:
        raise ValueError("No valid prompts could be built from the dense mask.")

    return prompts_by_slice


def mask_to_bbox_xyxy(mask2d):
    """
    Convert a binary 2D mask to XYXY bounding box format:
    [x_min, y_min, x_max, y_max]
    """
    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min = float(xs.min())
    x_max = float(xs.max())
    y_min = float(ys.min())
    y_max = float(ys.max())

    return np.array([x_min, y_min, x_max, y_max], dtype=np.float32)


def mask_to_center_point(mask2d):
    """
    Compute one center point [x, y] from a binary 2D mask.
    Uses the center of the mask's bounding box.
    """
    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_center = float((xs.min() + xs.max()) / 2.0)
    y_center = float((ys.min() + ys.max()) / 2.0)

    return np.array([x_center, y_center], dtype=np.float32)


def mask_to_negative_points_far_outside(mask2d, margin_px=10):
    """
    Generate negative points at fixed relative positions in the image,
    while keeping them safely outside the dense mask bounding box.

    Candidate positions are around:
        (25%,25%), (25%,75%), (75%,25%), (75%,75%)

    Points that fall inside or too close to the mask bounding box are discarded.
    """
    h, w = mask2d.shape

    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    # Expand bbox slightly to avoid placing a negative too close
    x_min_safe = max(0, x_min - margin_px)
    x_max_safe = min(w - 1, x_max + margin_px)
    y_min_safe = max(0, y_min - margin_px)
    y_max_safe = min(h - 1, y_max + margin_px)

    candidate_points = np.array([
        [0.25 * (w - 1), 0.25 * (h - 1)],
        [0.25 * (w - 1), 0.75 * (h - 1)],
        [0.75 * (w - 1), 0.25 * (h - 1)],
        [0.75 * (w - 1), 0.75 * (h - 1)],
    ], dtype=np.float32)

    valid_points = []
    for x, y in candidate_points:
        xi = int(round(x))
        yi = int(round(y))

        inside_safe_box = (
            x_min_safe <= xi <= x_max_safe and
            y_min_safe <= yi <= y_max_safe
        )

        inside_mask = mask2d[yi, xi] > 0

        if not inside_safe_box and not inside_mask:
            valid_points.append([x, y])

    if len(valid_points) == 0:
        return None

    return np.array(valid_points, dtype=np.float32)

In [3]:
img, conf_map_3d_dense = confidence_vis_pipeline(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    use_dense_mask=True,
    use_bbox=False,
    use_center_point=False,
    use_negative_background_points=False,   # NEW
    verbose=True,
    subject_nr=0
)


img, conf_map_3d_bbox = confidence_vis_pipeline(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    use_dense_mask=True,
    use_bbox=True,
    use_center_point=False,
    use_negative_background_points=False,   # NEW
    verbose=True,
    subject_nr=0
)


img, conf_map_3d_center = confidence_vis_pipeline(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    use_dense_mask=True,
    use_bbox=False,
    use_center_point=True,
    use_negative_background_points=True,   # NEW
    verbose=True,
    subject_nr=0
)

img, conf_map_3d_all = confidence_vis_pipeline(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    use_dense_mask=True,
    use_bbox=True,
    use_center_point=True,
    use_negative_background_points=True,   # NEW
    verbose=True,
    subject_nr=0
)

Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


Processing newAcq_050f229dc2bdb64c
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Adding prompt(s) on slice 27


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (default)...


...

propagate in video: 100%|██████████| 28/28 [00:08<00:00,  3.36it/s]


Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt
Processing newAcq_050f229dc2bdb64c
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Adding prompt(s) on slice 27
Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (default)...


propagate in video: 100%|██████████| 61/61 [00:14<00:00,  4.16it/s]


Backward propagation (default)...


propagate in video: 100%|██████████| 28/28 [00:09<00:00,  3.08it/s]


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider


def show_three_confidence_maps_inline(
    img_3d,
    conf1_3d,
    conf2_3d,
    conf3_3d,
    conf4_3d,
    title1="Confidence 1",
    title2="Confidence 2",
    title3="Confidence 3",
    title4="Confidence 4",
    gt_3d=None,
    pred1_3d=None,
    pred2_3d=None,
    pred3_3d=None,
    pred4_3d=None,
    alpha=0.5,
    conf_cmap="turbo",
):
    """
    Show three confidence maps side by side with one shared slice slider.

    Parameters
    ----------
    img_3d : np.ndarray
        3D grayscale image, shape (D, H, W)
    conf1_3d, conf2_3d, conf3_3d : np.ndarray
        3D confidence maps, shape (D, H, W)
    title1, title2, title3 : str
        Titles for the three panels
    gt_3d : np.ndarray or None
        Optional ground truth mask
    pred1_3d, pred2_3d, pred3_3d : np.ndarray or None
        Optional predicted binary masks for each panel
    alpha : float
        Overlay opacity
    conf_cmap : str
        Colormap for confidence overlay
    """
    D, H, W = img_3d.shape

    # Center crop to inner 25%
    crop_frac = 0.25
    h_half = int(H * crop_frac / 2)
    w_half = int(W * crop_frac / 2)

    cy, cx = H // 2, W // 2
    y0, y1 = cy - h_half, cy + h_half
    x0, x1 = cx - w_half, cx + w_half

    def _view(z):
        fig, axes = plt.subplots(1, 4, figsize=(24, 6))

        confs = [conf1_3d, conf2_3d, conf3_3d, conf4_3d]
        preds = [pred1_3d, pred2_3d, pred3_3d, pred4_3d]
        titles = [title1, title2, title3, title4]

        ims = []

        for ax, conf_3d, pred_3d, title in zip(axes, confs, preds, titles):
            img_c = img_3d[z][y0:y1, x0:x1]
            conf_c = conf_3d[z][y0:y1, x0:x1]

            ax.imshow(img_c, cmap="gray")
            im = ax.imshow(conf_c, cmap=conf_cmap, alpha=alpha, vmin=0.0, vmax=1.0)
            ims.append(im)

            if pred_3d is not None:
                pred_c = pred_3d[z][y0:y1, x0:x1]
                ax.contour(pred_c.astype(bool), levels=[0.5], linewidths=1)

            if gt_3d is not None:
                gt_c = gt_3d[z][y0:y1, x0:x1]
                ax.contour(gt_c.astype(bool), levels=[0.5], linewidths=1)

            ax.set_title(f"{title}\nSlice {z}")
            ax.axis("off")

        for ax, im in zip(axes, ims):
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Confidence")

        plt.tight_layout()
        plt.show()

    interact(
        _view,
        z=IntSlider(min=0, max=D - 1, step=1, value=D // 2, description="Slice"),
    )

In [ ]:
show_three_confidence_maps_inline(
    img_3d=img,
    conf1_3d=conf_map_3d_dense,
    conf2_3d=conf_map_3d_bbox,
    conf3_3d=conf_map_3d_center,
    conf4_3d=conf_map_3d_all,
    title1="Dense mask",
    title2="Bounding box",
    title3="Center point",
)

interactive(children=(IntSlider(value=44, description='Slice', max=87), Output()), _dom_classes=('widget-inter…